In [1]:
import tensorflow as tf

print(tf.__version__)

2.9.1


In [2]:
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, Input
from tensorflow.keras.activations import relu, softmax
from tensorflow.keras import Sequential

In [3]:
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import json

In [4]:
a = np.array([1, 2, 3])
print(a)

[1 2 3]


## Data Preparation

In [4]:
# leads to OOM when loading entire dataset into memory

# dataPath = "C:/Users/Wjunxion/Documents/datasets/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5"
# with h5py.File(dataPath, 'r') as f:
#     # 1. Check what datasets exist inside the file
#     print("Keys available:", list(f.keys()))
    
#     # 2. Access the target dataset by its key name
#     X = f['X'][:]
#     Y = f['Y'][:]
#     Z = f['Z'][:]
    
# print(X.shape)                     
# print(Y.shape)                         
# print(Z.shape)                         

In [ ]:
# lazy loading with `yield` and `tf.data.Dataset.from_generator`

file_path = "C:/Users/Wjunxion/Documents/datasets/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5"
train_size = 0

SEED=42
SHUFFLE_BUFFER = 512
BATCH_SIZE = 16
EPS = 1e-6

DATA_RATIO = 0.01 # dont use 2.5 million points on this laptop 
TRAIN_RATIO = 0.8


def make_split_indices(path, data_ratio=DATA_RATIO, train_ratio=TRAIN_RATIO):
    with h5py.File(path, "r") as hf:
        n = int(len(hf["X"]) * data_ratio)

    indices = np.arange(n)
    rng = np.random.default_rng(SEED)
    rng.shuffle(indices)

    split = int(n * train_ratio)

    train_indices = indices[:split]
    test_indices = indices[split:]

    return train_indices, test_indices

def hdf5_generator(path, indices, train=True):
    with h5py.File(path, "r") as hf:
        X = hf["X"]
        Y = hf["Y"]

        if train:
            print(f"Training data:", len(indices), "samples")
        else:
            print(f"Testing data:", len(indices), "samples")

        for i in indices:
            x = X[i].astype(np.float32)            # (1024,2)
            y = Y[i].astype(np.int32)              # (24,)

            yield x, y

def hdf5_generator_snr(path):
    with h5py.File(path, "r") as hf:
        Z = hf["Z"]

        for i in range(10000):
            z = Z[i].astype(np.int32)              # (1,)

            yield z

output_sig = (
    tf.TensorSpec(shape=(1024, 2), dtype=tf.float32, name="feature"),
    tf.TensorSpec(shape=(24,), dtype=tf.int32, name="label"),
)

output_sig_snr = tf.TensorSpec(shape=(1,), dtype=tf.int32, name="snr"),

def _prep(x, y):
    mean = tf.reduce_mean(x, axis=0, keepdims=True)   # (1,2)
    std = tf.math.reduce_std(x, axis=0, keepdims=True)
    x = (x - mean) / (std + EPS)
    x = tf.expand_dims(x, -1)                          # (1024,2,1) for convolution
    y = tf.cast(y, tf.float32)
                               
    return x, y

train_indices, test_indices = make_split_indices(file_path, data_ratio=DATA_RATIO, train_ratio=TRAIN_RATIO)
train_ds = tf.data.Dataset.from_generator(lambda: hdf5_generator(file_path, train_indices, train=True), 
                                          output_signature=output_sig)
train_ds = (
    train_ds.map(_prep, 
                 num_parallel_calls=tf.data.AUTOTUNE)
            .shuffle(SHUFFLE_BUFFER, seed=SEED)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE)
)

test_ds = tf.data.Dataset.from_generator(lambda: hdf5_generator(file_path, test_indices, train=False), 
                                         output_signature=output_sig)
test_ds = (
    test_ds.map(_prep, 
                num_parallel_calls=tf.data.AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(tf.data.AUTOTUNE)
)

ds_snr = tf.data.Dataset.from_generator(lambda: hdf5_generator_snr(file_path), 
                                        output_signature=output_sig_snr)
ds_snr = ds_snr.shuffle(SHUFFLE_BUFFER, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [9]:
for feature, label in train_ds.take(1):
    print("Feature:", feature.shape, feature[0])
    print("Label:", label.shape, label[0])
    # print("SNR:", snr.shape, snr[0])

Training data: 20447 samples
Feature: (16, 1024, 2, 1) tf.Tensor(
[[[-0.12071438]
  [ 0.14296253]]

 [[-0.40136755]
  [ 0.50072575]]

 [[ 0.7577169 ]
  [-0.5685583 ]]

 ...

 [[ 0.43521717]
  [ 0.70330167]]

 [[-1.3676862 ]
  [ 0.7828367 ]]

 [[-0.33742282]
  [-1.2833266 ]]], shape=(1024, 2, 1), dtype=float32)
Label: (16, 24) tf.Tensor([1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.], shape=(24,), dtype=float32)


In [8]:
fixed_classes = json.load(open("../data_labels/classes-fixed.json", 'r'))
print(fixed_classes)

['OOK', '4ASK', '8ASK', 'BPSK', 'QPSK', '8PSK', '16PSK', '32PSK', '16APSK', '32APSK', '64APSK', '128APSK', '16QAM', '32QAM', '64QAM', '128QAM', '256QAM', 'AM-SSB-WC', 'AM-SSB-SC', 'AM-DSB-WC', 'AM-DSB-SC', 'FM', 'GMSK', 'OQPSK']


In [9]:
# source: visualize modulations
# https://www.kaggle.com/code/suyashkumarbhagat/radio-ml-2018-data-visualisation

# Function to find the indices of a specific modulation type
def find_modulation_indices(modulation_name, classes_fixed):
    modulation_index = classes_fixed.index(modulation_name)
    indices = [i for i, label in enumerate(Y) if np.argmax(label) == modulation_index]
    return indices

def clean_classes(file_path):
    with open(file_path) as f:
        lines = f.readlines()
    classes_fixed = []
    for line in lines:
        if "|" in line and not line.strip().startswith("#"):
            parts = line.split("|")
            if len(parts) == 4:
                fixed_class = parts[3].strip().strip("'")
                if fixed_class and fixed_class not in ["Fixed"]:
                    classes_fixed.append(fixed_class)
    return classes_fixed

def find_modulation_snr_indices(modulation_name, snr_level, classes_fixed, Y, Z):
    modulation_index = classes_fixed.index(modulation_name)
    indices = [i for i, (label, snr) in enumerate(zip(Y, Z)) if np.argmax(label) == modulation_index and snr == snr_level]
    return indices

def plot_signals_by_modulation_snr(modulation_name, snr_level, num_samples):
    indices = find_modulation_snr_indices(modulation_name, snr_level, fixed_classes, Y, Z)
    if len(indices) < num_samples:
        print(f"Not enough samples for {modulation_name} with SNR {snr_level} dB. Found only {len(indices)} samples.")
        return

    samples = np.random.choice(indices, num_samples, replace=False)

    for sample in samples:
        plt.figure(figsize=(30, 3))

        I_phase = X[sample, :, 0]
        Q_phase = X[sample, :, 1]
        magnitude = np.sqrt((I_phase**2) + (Q_phase**2))

        plt.subplot(1, 3, 1)
        plt.plot(I_phase)
        plt.title(f'{modulation_name} - In-phase (I) (SNR: {snr_level} dB)')

        plt.subplot(1, 3, 2)
        plt.plot(Q_phase)
        plt.title(f'{modulation_name} - Quadrature-phase (Q) (SNR: {snr_level} dB)')

        plt.subplot(1, 3, 3)
        plt.plot(magnitude)
        plt.title(f'{modulation_name} - Magnitude (SNR: {snr_level} dB)')

        plt.tight_layout()
        plt.show()

In [10]:
#Modulations-['OOK', '4ASK', '8ASK', 'BPSK', 'QPSK', '8PSK', '16PSK', '32PSK', '16APSK', '32APSK', '64APSK', '128APSK', '16QAM', 
#              '32QAM', '64QAM', '128QAM', '256QAM', 'AM-SSB-WC', 'AM-SSB-SC', 'AM-DSB-WC', 'AM-DSB-SC', 'FM', 'GMSK', 'OQPSK']
"""modulation_type = '4ASK'  
snr_value = 20  
num_samples_to_plot = 2 """ 
#plot_signals_by_modulation_snr(modulation_type, snr_value, num_samples_to_plot)


# plot_signals_by_modulation_snr('4ASK', 24, 1)
# plot_signals_by_modulation_snr('128APSK', 24, 1)
# plot_signals_by_modulation_snr('16QAM', 24, 1)
# plot_signals_by_modulation_snr('256QAM', 24, 1)
# plot_signals_by_modulation_snr('AM-SSB-SC', 24, 1)
# plot_signals_by_modulation_snr('FM', 24, 1)
# plot_signals_by_modulation_snr('GMSK', 24, 1)
# plot_signals_by_modulation_snr('OQPSK', 24, 1)

"modulation_type = '4ASK'  \nsnr_value = 20  \nnum_samples_to_plot = 2 "

## Modelling Experiments

In [10]:
# tinyvgg model 
model_tinyvgg = Sequential([
    Input(shape=(1024, 2, 1)),
    Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation=relu),
    Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation=relu),
    MaxPool2D((2, 1)),
    Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation=relu),
    Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation=relu),
    MaxPool2D((2, 1)),
    Flatten(),
    Dense(24, activation=softmax),
], name="TinyVGG")

In [11]:
model_tinyvgg.summary()

Model: "TinyVGG"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 1024, 2, 16)       160       
                                                                 
 conv2d_1 (Conv2D)           (None, 1024, 2, 16)       2320      
                                                                 
 max_pooling2d (MaxPooling2D  (None, 512, 2, 16)       0         
 )                                                               
                                                                 
 conv2d_2 (Conv2D)           (None, 512, 2, 16)        2320      
                                                                 
 conv2d_3 (Conv2D)           (None, 512, 2, 16)        2320      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 256, 2, 16)       0         
 2D)                                                       

In [12]:
# BATCH_SIZE = 64
# BUFFER = 10000

# def _prep(x, y):
#     x = tf.cast(x, tf.float32)
#     # If model expects a channel dim (e.g., Conv2D), add it:
#     if tf.rank(x) == 2:
#         x = tf.expand_dims(x, -1)   # -> (1024, 2, 1)
#     y = tf.cast(y, tf.int32)
#     return x, y

# ds = tf.data.Dataset.from_tensor_slices((X, Y))
# ds = ds.map(_prep, num_parallel_calls=tf.data.AUTOTUNE)
# ds = ds.shuffle(BUFFER).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [13]:
model_tinyvgg.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
                      loss='categorical_crossentropy', 
                      metrics=['accuracy'])

In [14]:
history_tinyvgg = model_tinyvgg.fit(train_ds,
                                    epochs=5,
                                    batch_size=BATCH_SIZE,
                                    verbose=1)

Epoch 1/5


Training data: 20447 samples
1278/1278 [==============================] - 25s 19ms/step - loss: 0.0061 - accuracy: 0.9993
Epoch 2/5
Training data: 20447 samples
1278/1278 [==============================] - 21s 17ms/step - loss: 0.0000e+00 - accuracy: 1.0000
Epoch 3/5
Training data: 20447 samples
1278/1278 [==============================] - 25s 19ms/step - loss: 0.0000e+00 - accuracy: 1.0000
Epoch 4/5
Training data: 20447 samples
1278/1278 [==============================] - 21s 16ms/step - loss: 0.0000e+00 - accuracy: 1.0000
Epoch 5/5
Training data: 20447 samples
1278/1278 [==============================] - 21s 16ms/step - loss: 0.0000e+00 - accuracy: 1.0000


In [15]:
model_tinyvgg.evaluate(test_ds)

Testing data: 5112 samples
320/320 [==============================] - 3s 8ms/step - loss: 0.0000e+00 - accuracy: 1.0000


[0.0, 1.0]

In [16]:
# Convert the history dictionary to a DataFrame
history_df = pd.DataFrame(history_tinyvgg.history)

# Save to a CSV file
history_df.to_csv('../model_history/training_history.csv', index=False)

In [ ]:
plt.figure(figsize=(7, 10))
plt.plot(history_tinyvgg.history['accuracy'])

: 

In [ ]:
# tinyvgg model 
model_DenseNet = Sequential([
    Input(shape=(1024   , 2, 1)),
    Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation=relu),
    Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation=relu),
    MaxPool2D((2, 1)),
    Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation=relu),
    Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation=relu),
    MaxPool2D((2, 1)),
    Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation=relu),
    Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation=relu),
    MaxPool2D((2, 1)),
    Flatten(),
    Dense(24, activation=softmax),
], name="TinyVGG")